# Objectives and Loss Terms in NeuralRNN

This notebook is a hands-on guide to the **Objective** layer in NeuralRNN.

In NeuralRNN, a model is a discrete dynamical system with readout

$$\mathbf{z}_t = F_\theta(\mathbf{z}_{t-1}, \mathbf{x}_t), \qquad \mathbf{y}_t = G_\phi(\mathbf{z}_t).$$

The **model** implements $(F, G)$; the **objective** decides what "good" means.
Changing the objective changes the training paradigm without changing the model or the trainer.

**What you will learn**

1. How built-in objectives map onto Paradigm A (task optimization), Paradigm B (dynamics reconstruction), and behavioral fitting.
2. How to compose reusable loss terms, regularizers, and metrics from `neuralrnn.train.losses`.
3. How to use the `build_objective` factory and the `OBJECTIVE_REGISTRY`.
4. How to write and register your own custom objective.


In [ ]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../src')

import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from neuralrnn import (
    AutoConfig, AutoModel, Trainer, TrainingArguments,
    Objective, SupervisedObjective, RegularizedSupervisedObjective,
    TeacherForcingObjective, BehavioralObjective,
    ReconstructionObjective, ConstrainedSupervisedObjective,
    build_objective, register_objective, OBJECTIVE_REGISTRY,
    masked_mse, masked_cross_entropy, activity_l2, weight_l2,
    orthogonality_penalty, accuracy_classification, accuracy_general,
)
from neuralrnn.data import CustomDataset


# Trained demo models are load-first: rerunning this notebook loads the
# checkpoints under MODEL_DIR instead of retraining. Figures go to FIG_DIR.
MODEL_DIR = Path("./models/objectives")
FIG_DIR = Path("./figs/objectives")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


def train_or_load(model, ckpt_dir, train_fn):
    """Load `ckpt_dir` if it exists, else run `train_fn()` and save `model`."""
    ckpt_dir = Path(ckpt_dir)
    if (ckpt_dir / "config.json").exists():
        print(f"Loaded pretrained model from {ckpt_dir}")
        return AutoModel.from_pretrained(ckpt_dir)
    train_fn()
    model.save_pretrained(ckpt_dir)
    print(f"Saved checkpoint to {ckpt_dir}")
    return model


class SyntheticDataset:
    """Minimal dataset that returns the batch dict expected by NeuralRNN objectives."""

    def __init__(self, inputs, targets, mask=None, batch_size=32):
        self.inputs = inputs
        self.targets = targets
        self.mask = mask if mask is not None else torch.ones(inputs.shape[:2])
        self.batch_size = batch_size
        self.input_dim = inputs.shape[-1]
        self.output_dim = targets.shape[-1] if targets.dim() == 3 else 1

    def sample_batch(self):
        n = self.inputs.shape[0]
        idx = torch.randint(0, n, (self.batch_size,))
        return {
            "inputs": self.inputs[idx],
            "targets": self.targets[idx],
            "mask": self.mask[idx],
        }


print('Available objectives:', sorted(OBJECTIVE_REGISTRY.keys()))

## Notation: batch tensor shapes

All NeuralRNN objectives consume a batch dict with the following tensor shapes:

- `inputs`: `(B, T, K)` — `B` trials, `T` time steps per trial, `K` input features per step.
- `targets`: shape decides the task type:
  - **Classification**: `(B, T)` integer class indices in `[0, C - 1]`.
  - **Regression**: `(B, T, O)` continuous target values.
- `mask`: `(B, T)` or `(B, T, O)` — optional 0/1 weights marking which time steps count toward the loss.

Symbols used throughout this notebook:

| Symbol | Meaning | Where it appears |
|---|---|---|
| **B** | Batch size | Number of trials in one minibatch |
| **T** | Time steps | Trial length |
| **K** | Input dimension | `input_dim`, shape `inputs[..., K]` |
| **O** | Output dimension (regression) | `output_dim` for continuous targets, shape `targets[..., O]` |
| **C** | Number of classes (classification) | `output_dim` for categorical targets, shape `outputs[..., C]` |

The `SyntheticDataset` class above does not hard-code a task type; it only inspects `targets.dim()` to infer whether the target is continuous (`dim == 3`) or categorical (`dim == 2`). The actual loss is selected by the objective's `task_type` argument. This is why the same dataset class can serve both regression and classification examples.

## 1. The core contract

Every objective inherits from `Objective` and implements

```python
def compute_loss(self, model, batch) -> (loss: torch.Tensor, logs: dict):
    ...
```

`Trainer` calls this every step and backpropagates `loss`. The `logs` dict is printed/saved.
If the objective supports teacher-forcing annealing, it can also implement `set_forcing(alpha)`.

## 2. Supervised objectives (Paradigm A)

### 2.1 Classification

Batch: `inputs` $(B, T, K)$, `targets` $(B, T)$ integer class indices in $[0, C-1]$, optional `mask` $(B, T)$.

The model readout must have `output_dim = C` so that `outputs` has shape $(B, T, C)$ and can be interpreted as class logits.

We create a tiny synthetic classification task so the notebook runs in seconds.

In [ ]:
torch.manual_seed(0)
B, T, K, C = 256, 20, 4, 3
inputs_cls = torch.randn(B, T, K)
# Class label is determined by the sign of the first input channel summed over time
score = inputs_cls[:, :, 0].sum(dim=1, keepdim=True)
targets_cls = torch.where(score > 0.5, 0, torch.where(score < -0.5, 1, 2)).expand(-1, T)
mask_cls = torch.ones(B, T)

ds_cls = SyntheticDataset(inputs_cls, targets_cls, mask_cls, batch_size=32)

cfg_cls = AutoConfig.for_model(
    'ctrnn', input_dim=K, latent_dim=32, output_dim=C, dt=20, tau=100
)
model_cls = AutoModel.from_config(cfg_cls)

objective_cls = SupervisedObjective(task_type='classification')
args_cls = TrainingArguments(
    max_steps=200, learning_rate=1e-2, log_every=50, grad_clip_norm=1.0
)
model_cls = train_or_load(
    model_cls, MODEL_DIR / "ctrnn_classification",
    lambda: Trainer(model_cls, ds_cls, objective_cls, args_cls).train())

model_cls.eval()
with torch.no_grad():
    out = model_cls(inputs_cls)
    acc = accuracy_classification(out.outputs, targets_cls, mask_cls)
print(f'Classification accuracy: {acc:.4f}')

### 2.2 Regression

Batch: `inputs` $(B, T, K)$, `targets` $(B, T, O)$ continuous values, optional `mask` $(B, T)$ or $(B, T, O)$.

The model readout must have `output_dim = O` so that `outputs` has shape $(B, T, O)$.

Here we learn a simple delayed integration: the network must output a smoothed cumulative version of the input.

In [ ]:
torch.manual_seed(1)
B, T, K, O = 256, 30, 2, 1
inputs_reg = torch.randn(B, T, K)
targets_reg = inputs_reg[:, :, 0:1].cumsum(dim=1) * 0.1 + 0.1 * torch.randn(B, T, O)
mask_reg = torch.ones(B, T, O)

ds_reg = SyntheticDataset(inputs_reg, targets_reg, mask_reg, batch_size=32)

cfg_reg = AutoConfig.for_model(
    'ctrnn', input_dim=K, latent_dim=32, output_dim=O, dt=20, tau=100
)
model_reg = AutoModel.from_config(cfg_reg)

objective_reg = SupervisedObjective(task_type='regression')
args_reg = TrainingArguments(
    max_steps=200, learning_rate=1e-2, log_every=50, grad_clip_norm=1.0
)
model_reg = train_or_load(
    model_reg, MODEL_DIR / "ctrnn_regression",
    lambda: Trainer(model_reg, ds_reg, objective_reg, args_reg).train())

model_reg.eval()
with torch.no_grad():
    out = model_reg(inputs_reg)
    mse = masked_mse(out.outputs, targets_reg, mask_reg)
print(f'Regression MSE: {mse:.4f}')

### 2.3 A decision task can be trained as classification or regression

Decision-making tasks are conceptually classification problems (choose option A or B), but in cognitive-neuroscience modeling the choice is often encoded as a continuous signed signal and trained with MSE. The two formulations are equivalent as long as you convert the continuous readout back to a choice by taking its sign.

Below we generate the same binary decision data twice:

- **Classification formulation**: `targets` $(B, T)$ are integer class indices and the readout has dimension $C=2$.
- **Regression formulation**: `targets` $(B, T, 1)$ are signed scalars $+1$ / $-1$ and the readout has dimension $O=1$.

After training, both are evaluated on the same ground-truth choice. The regression version uses `accuracy_general`, which takes the sign of the masked mean output and compares it to the sign of the masked mean target. This is the pattern used in notebooks such as `07_lowrank_RNN_paradigmA.ipynb` and `08_lowrank_RNN_paradigmB.ipynb`.

In [ ]:
torch.manual_seed(6)

B, T, K = 512, 30, 3
inputs_dec = torch.randn(B, T, K)

# Common decision rule: integrate the first input channel and choose sign
evidence = inputs_dec[:, :, 0].sum(dim=1)           # (B,)
choice = (evidence > 0).long()                      # 1 = right, 0 = left

# --- Classification formulation ---
targets_dec_cls = choice.unsqueeze(1).expand(-1, T)  # (B, T)
mask_dec = torch.ones(B, T)

ds_dec_cls = SyntheticDataset(inputs_dec, targets_dec_cls, mask_dec, batch_size=32)

cfg_dec_cls = AutoConfig.for_model(
    'ctrnn', input_dim=K, latent_dim=32, output_dim=2, dt=20, tau=100
)
model_dec_cls = AutoModel.from_config(cfg_dec_cls)

obj_dec_cls = SupervisedObjective(task_type='classification')
model_dec_cls = train_or_load(
    model_dec_cls, MODEL_DIR / "ctrnn_decision_classification",
    lambda: Trainer(model_dec_cls, ds_dec_cls, obj_dec_cls,
                    TrainingArguments(max_steps=200, learning_rate=1e-2, log_every=50, grad_clip_norm=1.0)).train())

# --- Regression formulation ---
# Map left/right to signed scalar +1 / -1
targets_dec_reg = (2.0 * choice.float() - 1.0).unsqueeze(-1).unsqueeze(-1).expand(-1, T, 1)  # (B, T, 1)
mask_dec_reg = mask_dec.unsqueeze(-1).expand(-1, -1, 1)

ds_dec_reg = SyntheticDataset(inputs_dec, targets_dec_reg, mask_dec_reg, batch_size=32)

cfg_dec_reg = AutoConfig.for_model(
    'ctrnn', input_dim=K, latent_dim=32, output_dim=1, dt=20, tau=100
)
model_dec_reg = AutoModel.from_config(cfg_dec_reg)

obj_dec_reg = SupervisedObjective(task_type='regression')
model_dec_reg = train_or_load(
    model_dec_reg, MODEL_DIR / "ctrnn_decision_regression",
    lambda: Trainer(model_dec_reg, ds_dec_reg, obj_dec_reg,
                    TrainingArguments(max_steps=200, learning_rate=1e-2, log_every=50, grad_clip_norm=1.0)).train())

# Compare accuracies on the same ground-truth choice
model_dec_cls.eval()
model_dec_reg.eval()
with torch.no_grad():
    acc_cls = accuracy_classification(
        model_dec_cls(inputs_dec).outputs, targets_dec_cls, mask_dec)
    acc_reg = accuracy_general(
        model_dec_reg(inputs_dec).outputs, targets_dec_reg, mask_dec_reg)
print(f'Classification accuracy (argmax): {acc_cls:.4f}')
print(f'Regression -> sign accuracy:      {acc_reg:.4f}')

## 3. Regularized supervised objective

`RegularizedSupervisedObjective` adds optional penalties on top of the task loss:

- `activity_weight`: L2 firing-rate penalty $\lambda_h \mathbb{E}[\mathbf{h}^2]$
- `weight_weight`: L2 weight penalty $\lambda_W \sum_p \|p\|^2_2$
- `ortho_weight`: input/output orthogonality penalty

All weights default to `0.0`, so with no arguments it behaves like `SupervisedObjective`.

In [ ]:
torch.manual_seed(2)
model_reg2 = AutoModel.from_config(cfg_reg)

objective_reg2 = RegularizedSupervisedObjective(
    task_type='regression',
    activity_weight=1e-4,
    weight_weight=1e-5,
    ortho_weight=1e-4,  # safe to set even if the model does not expose the expected attrs
)
args_reg2 = TrainingArguments(
    max_steps=200, learning_rate=1e-2, log_every=50, grad_clip_norm=1.0
)
model_reg2 = train_or_load(
    model_reg2, MODEL_DIR / "ctrnn_regression_regularized",
    lambda: Trainer(model_reg2, ds_reg, objective_reg2, args_reg2).train())

model_reg2.eval()
with torch.no_grad():
    out = model_reg2(inputs_reg)
    mse = masked_mse(out.outputs, targets_reg, mask_reg)
print(f'Regularized regression MSE: {mse:.4f}')

## 4. Teacher forcing objective (Paradigm B)

`TeacherForcingObjective` is used for dynamical systems reconstruction.
At each step the predicted latent state is blended with the observed state:

$$\tilde{\mathbf{z}}_t = \alpha \mathbf{x}^{\text{obs}}_t + (1 - \alpha) \mathbf{z}^{\text{pred}}_t.$$

- $\alpha = 1$: pure teacher forcing.
- $\alpha = 0$: free running.
- Typical DSR setting: $\alpha = 0.1$ (sparse forcing).

We generate a simple damped oscillator trajectory for this demo.

In [ ]:
torch.manual_seed(3)
np.random.seed(3)

n_steps = 5000
dt = 0.05
t = np.arange(n_steps) * dt
trajectory = np.zeros((n_steps, 2), dtype=np.float32)
trajectory[0] = [1.0, 0.0]
for i in range(1, n_steps):
    x, y = trajectory[i-1]
    trajectory[i, 0] = x + dt * y
    trajectory[i, 1] = y + dt * (-0.5 * y - 4.0 * x)

ds_dsr = CustomDataset.from_arrays(
    trajectory, mode='timeseries', sequence_length=100, batch_size=16,
    test_fraction=0.2, seed=0
)

cfg_dsr = AutoConfig.for_model(
    'shallow_plrnn', latent_dim=2, hidden_dim=32, output_dim=2,
    autonomous=True
)
model_dsr = AutoModel.from_config(cfg_dsr)

objective_dsr = TeacherForcingObjective(alpha=0.1)
args_dsr = TrainingArguments(
    max_steps=500, learning_rate=1e-3, log_every=100, grad_clip_norm=1.0
)
model_dsr = train_or_load(
    model_dsr, MODEL_DIR / "plrnn_dsr",
    lambda: Trainer(model_dsr, ds_dsr, objective_dsr, args_dsr).train())

# Free rollout from the first validation state
model_dsr.eval()
with torch.no_grad():
    z0 = ds_dsr.test_set.X[0:1]
    gen = model_dsr.generate(z0, n_steps=200).numpy()

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(trajectory[:500, 0], trajectory[:500, 1], alpha=0.5, label='Data')
ax.plot(gen[:, 0], gen[:, 1], '--', label='Generated')
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(); ax.set_title('DSR rollout')
plt.tight_layout()
fig.savefig(FIG_DIR / "dsr_rollout.png", dpi=150)
plt.show()

## 5. Behavioral objective

`BehavioralObjective` fits next-action choice behavior. It is a masked cross-entropy objective with special handling for `output_h0=True` models (e.g. `tiny_rnn`).

Here we only show the API on synthetic discrete-choice data.

In [ ]:
torch.manual_seed(4)
B, T, K, n_actions = 128, 50, 3, 2
inputs_beh = torch.randn(B, T, K)
targets_beh = torch.randint(0, n_actions, (B, T))
mask_beh = torch.ones(B, T)

ds_beh = SyntheticDataset(inputs_beh, targets_beh, mask_beh, batch_size=32)

cfg_beh = AutoConfig.for_model(
    'ctrnn', input_dim=K, latent_dim=16, output_dim=n_actions, dt=20, tau=100
)
model_beh = AutoModel.from_config(cfg_beh)

objective_beh = BehavioralObjective()
args_beh = TrainingArguments(
    max_steps=200, learning_rate=1e-2, log_every=50, grad_clip_norm=1.0
)
model_beh = train_or_load(
    model_beh, MODEL_DIR / "ctrnn_behavioral",
    lambda: Trainer(model_beh, ds_beh, objective_beh, args_beh).train())

model_beh.eval()
with torch.no_grad():
    out = model_beh(inputs_beh)
    acc = accuracy_classification(out.outputs, targets_beh, mask_beh)
print(f'Behavioral choice accuracy: {acc:.4f}')

## 6. Reusable loss terms, regularizers, and metrics

All built-in objectives delegate to functions in `neuralrnn.train.losses`. You can use these same functions when writing custom objectives.

In [ ]:
B, T, O = 8, 12, 2
out = torch.randn(B, T, O)
tgt = torch.randn(B, T, O)
mask = torch.ones(B, T, 1)

print('masked_mse:', masked_mse(out, tgt, mask).item())
print('masked_cross_entropy:', masked_cross_entropy(
    torch.randn(B, T, 3), torch.randint(0, 3, (B, T))
).item())

states = torch.randn(B, T, O)
print('activity_l2:', activity_l2(states, mask).item())
print('weight_l2:', weight_l2(model_reg, patterns=['h2h', 'input2h']).item())
print('accuracy_general:', accuracy_general(out, tgt, mask).item())

## 7. The `build_objective` factory

Objectives are registered by name. This mirrors `AutoConfig` / `AutoModel` and lets you instantiate objectives from strings (e.g. in config files).

In [ ]:
obj1 = build_objective('supervised', task_type='classification')
obj2 = build_objective('regularized_supervised', task_type='regression', activity_weight=1e-4)
obj3 = build_objective('teacher_forcing', alpha=0.1)
print(obj1, obj2, obj3, sep='\n')

## 8. Writing a custom objective

The minimal contract is `compute_loss(model, batch)`. Below we implement an objective that combines a task loss with an L2 activity penalty and a weight-decay term, but only on recurrent weights.

In [ ]:
from neuralrnn.train.objectives.base import Objective
from neuralrnn.train.losses import masked_mse, activity_l2, weight_l2

class RecurrentRegularizedObjective(Objective):
    """Task MSE + L2 activity + L2 recurrent weight penalty."""

    def __init__(self, activity_weight=1e-4, weight_weight=1e-5):
        self.activity_weight = activity_weight
        self.weight_weight = weight_weight

    def compute_loss(self, model, batch):
        out = model(batch['inputs'])
        task_loss = masked_mse(out.outputs, batch['targets'], batch.get('mask'))
        reg = activity_l2(out.states, batch.get('mask'))
        wreg = weight_l2(model, patterns=['h2h.weight'])
        loss = task_loss + self.activity_weight * reg + self.weight_weight * wreg
        return loss, {
            'loss': loss.item(),
            'task': task_loss.item(),
            'activity': reg.item(),
            'weight': wreg.item(),
        }

torch.manual_seed(5)
model_custom = AutoModel.from_config(cfg_reg)
args_custom = TrainingArguments(
    max_steps=200, learning_rate=1e-2, log_every=50, grad_clip_norm=1.0
)
model_custom = train_or_load(
    model_custom, MODEL_DIR / "ctrnn_custom_objective",
    lambda: Trainer(model_custom, ds_reg, RecurrentRegularizedObjective(), args_custom).train())

model_custom.eval()
with torch.no_grad():
    out = model_custom(inputs_reg)
    mse = masked_mse(out.outputs, targets_reg, mask_reg)
print(f'Custom objective MSE: {mse:.4f}')

### 8.1 Registering the custom objective

Use `@register_objective` so it becomes available through `build_objective`.

In [ ]:
@register_objective('recurrent_regularized')
class RegisteredRecurrentRegularizedObjective(RecurrentRegularizedObjective):
    pass

print('recurrent_regularized' in OBJECTIVE_REGISTRY)
obj_custom = build_objective('recurrent_regularized', activity_weight=1e-3)
print(obj_custom)

## 9. Summary

| Concept | API | Use when |
|---|---|---|
| Classification | `SupervisedObjective('classification')` | Discrete choice tasks with integer targets `(B, T)` and logits `(B, T, C)` |
| Regression | `SupervisedObjective('regression')` | Continuous outputs `(B, T, O)`; can also be used for binary decisions if targets are signed scalars and accuracy is computed with `accuracy_general` |
| Regularized task | `RegularizedSupervisedObjective(...)` | Task + activity/weight/ortho penalties |
| Dynamics reconstruction | `TeacherForcingObjective(alpha=0.1)` | Reconstruct neural/behavioral trajectories |
| Behavioral fitting | `BehavioralObjective()` | Next-choice NLL |
| Custom | subclass `Objective` + `compute_loss` | Anything else |

Key takeaways:

1. `Trainer` is paradigm-agnostic; the objective decides the paradigm.
2. The same `SyntheticDataset` can be used for classification or regression — the task type is determined by the shape of `targets` and the `task_type` argument passed to the objective.
3. Decision tasks can be trained either with integer class labels and cross-entropy, or with signed continuous targets and MSE followed by sign-thresholding.
4. Reusable loss terms live in `neuralrnn.train.losses`.
5. Use `build_objective` / `register_objective` for config-driven training.
6. Custom objectives only need to implement `compute_loss(model, batch)`.